# PhoBERT loss comparison - 5-fold cross-validation (Kaggle)

**Before running**: Accelerator **GPU**, Internet **On**. Code and processed data are cloned from GitHub (push your latest changes first).

Five training losses on the same 5 folds, everything else identical (5 epochs, layer-wise learning rate decay 0.9, no R-Drop). The checkpoint of each fold is chosen on validation **weighted cross-entropy** whatever the training loss, so the runs are directly comparable.

| run | training loss |
|---|---|
| `cross_entropy` | class-weighted cross-entropy (the baseline) |
| `label_smoothing` | + label smoothing 0.1 |
| `focal` | focal loss, gamma 2 |
| `gce` | generalised cross-entropy, q 0.7 |
| `supcon` | 0.5 x cross-entropy + 0.5 x supervised contrastive (temperature 0.3) |

In [ ]:
!pip install -q pyvi

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/Zaidve/TMG_project_re.git"
CLONE_DIR = Path("/tmp/TMG_project_re")  # outside /kaggle/working so it is not saved as output
ON_KAGGLE = Path("/kaggle/input").exists()
SEARCH_ROOTS = [Path("/kaggle/input"), CLONE_DIR]

def find(pattern):
    return [hit for root in SEARCH_ROOTS if root.exists() for hit in sorted(root.rglob(pattern))]

def find_parent(pattern, levels):
    """Directory `levels` above the first matching file in an attached dataset or the clone."""
    hits = find(pattern)
    if not hits:
        raise FileNotFoundError(f"{pattern} not found under {[str(r) for r in SEARCH_ROOTS]}")
    return hits[0].parents[levels]

if ON_KAGGLE and not find("trainer/train_phobert_cv.py"):
    !git clone --depth 1 {REPO_URL} {CLONE_DIR}

if ON_KAGGLE:
    CODE_DIR = find_parent("trainer/train_phobert_cv.py", 1)
    DATA_DIR = find_parent("train.jsonl", 0)
    OUT_DIR = Path("/kaggle/working/phobert_loss")
else:  # local run from the notebooks/ folder
    CODE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DATA_DIR = CODE_DIR / "data" / "processed"
    OUT_DIR = CODE_DIR / "outputs" / "phobert_loss"

sys.path.insert(0, str(CODE_DIR))
print("code:", CODE_DIR)
print("data:", DATA_DIR)
print("out: ", OUT_DIR)

In [ ]:
import torch

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Train

About 6 minutes per run on a T4, 30 minutes in total. Remove entries from `RUNS` to skip them.

In [ ]:
from trainer.train_phobert_cv import CVConfig, run

BASE = dict(
    data_dir=str(DATA_DIR),
    n_folds=5,
    max_length=128,
    use_lead=True,
    batch_size=32,
    lr=2e-5,
    epochs=5,
    layer_decay=0.9,
    weighted_loss=True,
    patience=3,
    monitor="loss",
    seed=42,
    keep_checkpoints=False,
    limit=0,              # set to e.g. 64 for a quick smoke test
)
RUNS = {
    "cross_entropy": dict(),
    "label_smoothing": dict(label_smoothing=0.1),
    "focal": dict(focal_gamma=2.0),
    "gce": dict(gce_q=0.7),
    "supcon": dict(supcon_weight=0.5, supcon_temperature=0.3),
}

results = {}
for name, overrides in RUNS.items():
    print(f"################ {name} ################")
    results[name] = run(CVConfig(**BASE, **overrides, out_dir=str(OUT_DIR / name)))

## Compare

Judge by `oof_f1` (3,067 articles). Fold-to-fold spread is about one point, so smaller differences are noise. `oof_logloss` is the plain (unweighted) log loss of the out-of-fold probabilities: lower means better-calibrated confidence.

In [ ]:
import numpy as np
import pandas as pd

rows = {}
for name, r in results.items():
    s, oof = r["summary"], r["oof_predictions"]
    y = (oof["label"] == "clickbait").to_numpy()
    p = oof["prob_clickbait"].clip(1e-6, 1 - 1e-6).to_numpy()
    rows[name] = {
        "oof_f1": s["oof"]["f1"], "oof_precision": s["oof"]["precision"], "oof_recall": s["oof"]["recall"],
        "oof_logloss": -np.mean(np.where(y, np.log(p), np.log(1 - p))),
        "fold_f1_std": s["fold_val_std"]["f1"],
        "single_test_f1": s["single_model_test_mean"]["f1"], "ensemble_test_f1": s["ensemble_test"]["f1"],
    }
comparison = pd.DataFrame(rows).T
display(comparison.round(4))
if "cross_entropy" in comparison.index:
    print("difference from cross_entropy:")
    display((comparison - comparison.loc["cross_entropy"]).drop(index="cross_entropy").round(4))

SHOW = comparison["oof_f1"].idxmax()   # run detailed below; set a name to override
result = results[SHOW]
print("showing details for:", SHOW)

## Per-fold results

`val_*` is each model on its own held-out fold; `test_*` is the same model alone on the test split.

In [ ]:
folds = result["fold_metrics"]
display(folds.round(4))
folds.agg(["mean", "std"]).round(4)

## Cross-validated score vs single model vs ensemble

In [ ]:
summary = pd.DataFrame(result["summary"]).T.loc[["oof", "single_model_test_mean", "single_model_test_std", "ensemble_test"]]
summary.index = ["out-of-fold (pooled train+val)", "single model on test (mean)", "single model on test (std)", "ensemble on test"]
summary.round(4)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar([f"fold {i}" for i in folds.index], folds["test_f1"], color="#9ecae1", label="single model")
ax.axhline(result["summary"]["ensemble_test"]["f1"], color="#E45756", linestyle="--", label="ensemble")
ax.set_ylim(max(0, folds["test_f1"].min() - 0.1), 1)
ax.set_ylabel("test F1 (clickbait)")
ax.legend(frameon=False)
fig.tight_layout()

## Ensemble on test

In [ ]:
test = result["test_predictions"]
pd.crosstab(test["label"], test["pred"], rownames=["actual"], colnames=["predicted"], margins=True)

### Threshold

Chosen on the out-of-fold predictions (never on test), then applied to the ensemble.

In [ ]:
from trainer.trainer import classification_metrics

oof = result["oof_predictions"]
oof_y, test_y = (oof["label"] == "clickbait").to_numpy(), (test["label"] == "clickbait").to_numpy()

rows = []
for t in np.arange(0.2, 0.81, 0.05):
    o = classification_metrics(oof_y, (oof["prob_clickbait"] >= t).to_numpy())
    e = classification_metrics(test_y, (test["prob_clickbait"] >= t).to_numpy())
    rows.append({"threshold": round(t, 2), "oof_precision": o["precision"], "oof_recall": o["recall"], "oof_f1": o["f1"],
                 "test_precision": e["precision"], "test_recall": e["recall"], "test_f1": e["f1"]})
thresholds = pd.DataFrame(rows).set_index("threshold")
print("best threshold on out-of-fold F1:", thresholds["oof_f1"].idxmax())
thresholds.round(3)

### Out-of-fold results per source and category

Computed on all pooled articles, so the groups are large enough to compare.

In [ ]:
def by_group(table, col):
    rows = {}
    for name, g in table.groupby(col):
        m = classification_metrics((g["label"] == "clickbait").to_numpy(), (g["pred"] == "clickbait").to_numpy())
        rows[name] = {"n": len(g), "clickbait_rate": (g["label"] == "clickbait").mean(), **m}
    return pd.DataFrame(rows).T.round(3).sort_values("n", ascending=False)

by_group(oof, "source")

In [ ]:
by_group(oof, "category")

### Most confident out-of-fold mistakes

Each prediction comes from a model that never saw the article, so confident disagreements are good candidates for a label review.

In [ ]:
pd.set_option("display.max_colwidth", 140)
wrong = oof[oof["label"] != oof["pred"]].copy()
wrong["confidence"] = (wrong["prob_clickbait"] - 0.5).abs() + 0.5
print(len(wrong), "out-of-fold mistakes;", (wrong["confidence"] >= 0.9).sum(), "with confidence >= 0.9")
wrong.sort_values("confidence", ascending=False).head(30)[["id", "source", "title", "label", "pred", "prob_clickbait"]]

## Saved files

`summary.json`, `fold_metrics.csv`, `oof_predictions.csv`, `test_predictions.csv`, `tokenizer/`, per-fold `history.json`. On Kaggle they are under `/kaggle/working/phobert_loss/<run name>` and appear in the Output tab after **Save Version**.

In [ ]:
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1e6:8.1f} MB  {p.relative_to(OUT_DIR)}")